# Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset

In [5]:
# Rating Data
data_df = pd.read_csv('./data/u.data', sep="\t", header=None)
data_df.columns = ['user id', 'movie id', 'rating', 'timestamp']

# User Data
user_df = pd.read_csv('./data/u.user', sep="|", encoding='latin-1', header=None)
user_df.columns = ['user id', 'age', 'gender', 'occupation', 'zip code']

# Movie Data
item_df = pd.read_csv('./data/u.item', sep="|", encoding='latin-1', header=None)
item_df.columns = ['movie id', 'movie title' ,'release date','video release date', 'IMDb URL', 'unknown', 'Action', 
                'Adventure', 'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 
                'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

In [41]:
genre_list = pd.read_csv('./data/u.genre', sep="|", encoding='latin-1', header=None)
genre_list = genre_list[0].tolist()

In [6]:
data_df.head()

,user id,movie id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [7]:
data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user id    100000 non-null  int64
 1   movie id   100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB


In [8]:
user_df.head()

,user id,age,gender,occupation,zip code
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213


In [9]:
user_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 943 entries, 0 to 942
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user id     943 non-null    int64 
 1   age         943 non-null    int64 
 2   gender      943 non-null    object
 3   occupation  943 non-null    object
 4   zip code    943 non-null    object
dtypes: int64(2), object(3)
memory usage: 37.0+ KB


In [12]:
item_df.head().T

,0,1,2,3,4
movie id,1,2,3,4,5
movie title,Toy Story (1995),GoldenEye (1995),Four Rooms (1995),Get Shorty (1995),Copycat (1995)
release date,01-Jan-1995,01-Jan-1995,01-Jan-1995,01-Jan-1995,01-Jan-1995
video release date,NaN,NaN,NaN,NaN,NaN
IMDb URL,http://us.imdb.com/M/title-exact?Toy%20Story%2...,http://us.imdb.com/M/title-exact?GoldenEye%20(...,http://us.imdb.com/M/title-exact?Four%20Rooms%...,http://us.imdb.com/M/title-exact?Get%20Shorty%...,http://us.imdb.com/M/title-exact?Copycat%20(1995)
unknown,0,0,0,0,0
Action,0,1,0,1,0
Adventure,0,1,0,0,0
Animation,1,0,0,0,0
Children's,1,0,0,0,0


In [11]:
item_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1682 entries, 0 to 1681
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   movie id            1682 non-null   int64  
 1   movie title         1682 non-null   object 
 2   release date        1681 non-null   object 
 3   video release date  0 non-null      float64
 4   IMDb URL            1679 non-null   object 
 5   unknown             1682 non-null   int64  
 6   Action              1682 non-null   int64  
 7   Adventure           1682 non-null   int64  
 8   Animation           1682 non-null   int64  
 9   Children's          1682 non-null   int64  
 10  Comedy              1682 non-null   int64  
 11  Crime               1682 non-null   int64  
 12  Documentary         1682 non-null   int64  
 13  Drama               1682 non-null   int64  
 14  Fantasy             1682 non-null   int64  
 15  Film-Noir           1682 non-null   int64  
 16  Horror

# EDA

In [24]:
def find_ratings_by_users(user_ids, rating_df):
    output = []
    for user_id in user_ids:
        output.append(rating_df[rating_df["user id"] == user_id])
    return pd.concat(output)
def find_movie_by_id(id_, select_col, movie_df):
    return movie_df.iloc[id_-1][select_col]

In [36]:
# Task: Top 25 movies with highest rate from programmers
programmer_users = user_df[user_df["occupation"] == "programmer"]
programmer_users_ratings = find_ratings_by_users(programmer_users["user id"], data_df)
top25_programmer_movies = programmer_users_ratings.groupby("movie id")["rating"].mean().sort_values(ascending=False).iloc[:25]
top25_programmer_movies_title = [find_movie_by_id(_id, "movie title", item_df) for _id in top25_programmer_movies.index]
pd.DataFrame({"Movie Title": top25_programmer_movies_title, "Average Score": top25_programmer_movies.values})

,Movie Title,Average Score
0,"Little Princess, The (1939)",5.000000
1,8 Seconds (1994),5.000000
2,Late Bloomers (1996),5.000000
3,Love in the Afternoon (1957),5.000000
4,Faithful (1996),5.000000
5,Total Eclipse (1995),5.000000
6,Love & Human Remains (1993),5.000000
7,Beautiful Thing (1996),5.000000
8,Trust (1990),5.000000
9,Meet John Doe (1941),5.000000


In [48]:
# Task: Top 7 genres with highest rate movies
ratings_and_movie_data = pd.merge(programmer_users_ratings[['user id', 'movie id', 'rating']], item_df, on='movie id')
result = {"Genre": [], "Rating": []}
for genre in genre_list:
    movies_with_genre = programmer_users_ratings_and_movie_data[programmer_users_ratings_and_movie_data[genre] == 1]
    result["Genre"].append(genre)
    result["Rating"].append(movies_with_genre["rating"].mean())

pd.DataFrame(result).sort_values(by=["Rating"], ascending=False).reset_index(drop=True).iloc[:7]

,Genre,Rating
0,unknown,4.000000
1,War,3.888742
2,Film-Noir,3.886525
3,Drama,3.755714
4,Western,3.746914
5,Documentary,3.725490
6,Sci-Fi,3.680377


In [50]:
# Task: Top 7 genres with highest rate from programmers
programmer_users = user_df[user_df["occupation"] == "programmer"]
programmer_users_ratings = find_ratings_by_users(programmer_users["user id"], data_df)
programmer_users_ratings_and_movie_data = pd.merge(programmer_users_ratings[['user id', 'movie id', 'rating']], item_df, on='movie id')
result = {"Genre": [], "Movie Title": [], "Rating": []}
for genre in genre_list:
    movies_with_genre = programmer_users_ratings_and_movie_data[programmer_users_ratings_and_movie_data[genre] == 1]
    top_movies_within_genre = movies_with_genre.groupby("movie title", as_index=False)["rating"].mean().sort_values(by=["rating"], ascending=False)
    result["Genre"].append(genre)
    result["Movie Title"].append(top_movies_within_genre["movie title"].iloc[0])
    result["Rating"].append(top_movies_within_genre["rating"].iloc[0])

pd.DataFrame(result).sort_values(by=["Rating"]).reset_index(drop=True).iloc[:7]

,Genre,Movie Title,Rating
0,unknown,unknown,4.000000
1,Fantasy,"20,000 Leagues Under the Sea (1954)",4.500000
2,Horror,"Prophecy, The (1995)",4.500000
3,Western,High Noon (1952),4.500000
4,Sci-Fi,Star Wars (1977),4.540000
5,Adventure,"Old Man and the Sea, The (1958)",4.666667
6,War,Casablanca (1942),4.666667
